In [ ]:
import pandas as pd
import numpy as np

# Membaca file CSV
try:
    df = pd.read_csv('iotsensordata.csv')
    print("✅ Dataset berhasil dimuat!\n")
except FileNotFoundError:
    print("❌ Dataset Gagal Dimuat")

# Menampilkan 5 data teratas
display(df.head())


✅ Dataset berhasil dimuat!



,Soil Moisture,Temperature,Soil Humidity,Time,Air temperature (C),Wind speed (Km/h),Air humidity (%),Wind gust (Km/h),Pressure (KPa),ph,rainfall,N,P,K,Status
0,54,22,70,21,19.52,2.13,55.04,6.30,101.50,6.502985,202.935536,90.0,42.0,43.0,ON
1,12,20,40,104,19.49,2.01,55.17,10.46,101.50,7.038096,226.655537,85.0,58.0,41.0,OFF
2,34,26,35,62,19.47,1.90,55.30,14.63,101.51,7.840207,263.964248,60.0,55.0,44.0,ON
3,7,44,44,93,19.54,2.28,54.20,16.08,101.51,6.980401,242.864034,74.0,35.0,40.0,OFF
4,50,38,23,92,19.61,2.66,53.09,17.52,101.51,7.628473,262.717340,78.0,42.0,42.0,OFF


In [ ]:
# 1. Menghapus spasi tersembunyi pada nama kolom bawaan dataset
df.columns = df.columns.str.strip()

# 2. Mengambil hanya 4 kolom yang relevan dengan riset (Soil Moistur, Temperature, Humidity, Output)
kolom_riset = ['Soil Moisture', 'Temperature', 'Soil Humidity', 'Status']
df_clean = df[kolom_riset].copy()

# 3. Mengubah nama kolom menjadi format yang rapi dan standar
df_clean.columns = ['Soil_Moisture', 'Temperature', 'Air_Humidity', 'Status']

# 4. Mengubah teks pada kolom Status menjadi angka biner (1 = Siram, 0 = Mati)
df_clean['Status'] = df_clean['Status'].astype(str).str.strip().str.upper()
df_clean['Status'] = df_clean['Status'].apply(lambda x: 1 if x in ['ON', 'SIRAM', 'YES', '1'] else 0)

# 5. Membuang baris data jika ada nilai yang kosong (NaN)
df_clean = df_clean.dropna()

print("✅ Seleksi data selesai!")
print(f"Dimensi data sekarang: {df_clean.shape[0]} baris & {df_clean.shape[1]} kolom.\n")

# Menampilkan 5 data teratas
display(df_clean.head())


✅ Seleksi data selesai!
Dimensi data sekarang: 100000 baris & 4 kolom.



,Soil_Moisture,Temperature,Air_Humidity,Status
0,54,22,70,1
1,12,20,40,0
2,34,26,35,1
3,7,44,44,0
4,50,38,23,0


In [ ]:
# =====================================================================
# CELL 2B: PENYEIMBANGAN DATA (DATA BALANCING VIA UNDERSAMPLING)
# =====================================================================

# 1. Pisahkan data menjadi kelompok Status 0 dan Status 1
df_status_0 = df_clean[df_clean['Status'] == 0]
df_status_1 = df_clean[df_clean['Status'] == 1]

# 2. Hitung jumlah data masing-masing kelompok
jumlah_0 = len(df_status_0)
jumlah_1 = len(df_status_1)

print("=== JUMLAH DATA SEBELUM BALANCING ===")
print(f"Jumlah Data Pompa MATI  (0): {jumlah_0} baris")
print(f"Jumlah Data Pompa SIRAM (1): {jumlah_1} baris")

# 3. Ambil jumlah terkecil sebagai target balancing
jumlah_target = min(jumlah_0, jumlah_1)
print(f"-> Target penyeimbangan: diambil {jumlah_target} baris untuk masing-masing kelas.\n")

# 4. Lakukan pemotongan acak (sample) pada data yang lebih banyak
df_status_0_balanced = df_status_0.sample(n=jumlah_target, random_state=42)
df_status_1_balanced = df_status_1.sample(n=jumlah_target, random_state=42)

# 5. Gabungkan kembali data yang sudah seimbang
df_balanced = pd.concat([df_status_0_balanced, df_status_1_balanced])

# 6. Mengocok ulang urutan data agar acak saat dibaca AI nanti
df_clean = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("=== JUMLAH DATA SETELAH BALANCING ===")
print(df_clean['Status'].value_counts())


=== JUMLAH DATA SEBELUM BALANCING ===
Jumlah Data Pompa MATI  (0): 46350 baris
Jumlah Data Pompa SIRAM (1): 53650 baris
-> Target penyeimbangan: diambil 46350 baris untuk masing-masing kelas.

=== JUMLAH DATA SETELAH BALANCING ===
Status
0    46350
1    46350
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

# 1. Memisahkan Fitur Sensor (X) dan Target Keputusan (y)
X = df_clean[['Soil_Moisture', 'Temperature', 'Air_Humidity']]
y = df_clean['Status']

# 2. Membagi data menjadi porsi Training (80%) dan Testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Jumlah data Latihan (X_train) : {X_train.shape[0]} baris")
print(f"Jumlah data Ujian (X_test)    : {X_test.shape[0]} baris")


Jumlah data Latihan (X_train) : 74160 baris
Jumlah data Ujian (X_test)    : 18540 baris


In [ ]:
from sklearn.tree import DecisionTreeClassifier

print("⏳ Training model Decision Tree...")

# Membuat objek model dengan batasan kedalaman pohon (max_depth=4)
# Batasan ini krusial agar memori ESP32-S3 Anda nanti tidak kewalahan
model_irigasi = DecisionTreeClassifier(max_depth=8, random_state=42)

# Proses Training (AI mempelajari pola data sensor terhadap status pompa)
model_irigasi.fit(X_train, y_train)

print("Model DecisionTree telah selesai dilatih")


⏳ Training model Decision Tree...
Model DecisionTree telah selesai dilatih


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Menyuruh AI menebak data ujian (X_test) yang belum pernah ia lihat
y_pred = model_irigasi.predict(X_test)

# Menghitung persentase keakuratan jawaban AI dibandingkan kunci jawaban asli (y_test)
akurasi = accuracy_score(y_test, y_pred)

print("✅ Evaluasi model selesai!\n")
print("==================================================")
print(f"AKURASI MODEL: {akurasi * 100:.2f} %")
print("==================================================\n")

print("Detail Analisis Performa AI:")
print(classification_report(y_test, y_pred))


✅ Evaluasi model selesai!

AKURASI MODEL: 73.32 %

Detail Analisis Performa AI:
              precision    recall  f1-score   support

           0       0.74      0.71      0.73      9199
           1       0.73      0.76      0.74      9341

    accuracy                           0.73     18540
   macro avg       0.73      0.73      0.73     18540
weighted avg       0.73      0.73      0.73     18540



In [ ]:
def konversi_ke_cpp_fixed(tree, feature_names):
    tree_ = tree.tree_
    feature_name = [
        feature_names[i] if i != -2 else "undefined!"
        for i in tree_.feature
    ]

    cpp_code = []
    cpp_code.append("// File otomatis di-generate oleh Python untuk ESP32-S3")
    cpp_code.append("#ifndef MODEL_AI_IRIGASI_H")
    cpp_code.append("#define MODEL_AI_IRIGASI_H\n")
    cpp_code.append("int prediksi_penyiraman(float Soil_Moisture, float Temperature, float Air_Humidity) {")

    def recurse(node, depth):
        indent = "    " * depth
        if tree_.feature[node] != -2:
            name = feature_name[node]
            threshold = tree_.threshold[node]
            cpp_code.append(f"{indent}if ({name} <= {threshold:.2f}) {{")
            recurse(tree_.children_left[node], depth + 1)
            cpp_code.append(f"{indent}}} else {{")
            recurse(tree_.children_right[node], depth + 1)
            cpp_code.append(f"{indent}}}")
        else:
            # PERBAIKAN: Mengambil indeks dengan jumlah sampel terbanyak di leaf node
            # nilai_node[0][0] adalah jumlah data Kelas 0, nilai_node[0][1] adalah jumlah data Kelas 1
            nilai_node = tree_.value[node][0]
            keputusan = 0 if nilai_node[0] > nilai_node[1] else 1
            cpp_code.append(f"{indent}return {keputusan};")

    recurse(0, 1)

    cpp_code.append("}")
    cpp_code.append("#endif")
    return "\n".join(cpp_code)

nama_fitur = ['Soil_Moisture', 'Temperature', 'Air_Humidity']
kode_cpp_final = konversi_ke_cpp_fixed(model_irigasi, nama_fitur)

# Menyimpan ulang file
with open("model_decisiiontree_irigasi.h", "w") as f:
    f.write(kode_cpp_final)
print("\n💾 File .h berhasil disimpan")



💾 File .h berhasil disimpan
